# Lab 1 : First call + The no-memory surprise + Carrying history

*W1 LLM Foundations  ·  Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.


## What we are achieving in this lab

**Objective.** Get a real LLM running on your laptop, make your first Python call to it, and then see one of the most important surprises about how LLMs work: they have NO memory between calls. Then we fix it by carrying the history ourselves.

**Prerequisites.** A laptop with 8 GB RAM. No AI experience. No paid accounts.

**What you will do.**

1. Install Ollama on Mac, Linux, or Windows. See [README.md](./README.md).
2. Pull the `llama3.2:1b` model (about 1.3 GB, runs on CPU, no GPU needed).
3. Test the model from the terminal to confirm it works. See [README.md](./README.md)
4. Make your first Python call to the model with a greeting. See [README.md](./README.md)
5. Use the Python SDK to ask the model "what is my name?" and watch the model have NO idea.
6. Fix the memory problem by sending the full conversation history in the messages list.

**What you should see.**

- Step 6: a friendly greeting from the model.
- Step 7: the model has NO IDEA who you are. Every call is independent.
- Step 8: the model answers correctly because YOUR code carried the history.

**Why this matters.** LLMs are stateless. The "memory" you feel in ChatGPT is your app resending the whole conversation every turn. Every message you send pays for ALL prior turns. This is the foundation of how every chat product works. You will meet the 4 message roles this week: `system`, `user`, `assistant`, `tool`.


### **Step 4. Use the Python SDK to call the model**

The `ollama` Python package talks to the ollama background service you  tested from the CLI (see the [README.md](./README.md)) file. The downloaded model is called from the Python SDK instead.

If you have not followed the instructions to install the ollama python SDK, please refer to the [README.md](./README.md) file before move forward.

In [13]:
import ollama

**The 3 message roles (worth understanding before Step 6)**

Every LLM chat call uses a list of messages. Each message has a **role**. There are three main roles:

| Role | Who wrote it | What it does |
|------|--------------|--------------|
| `system` | You (the app author) | The standing instructions. "You are a polite support agent." Sets tone, rules, and persona. Comes FIRST in the messages list. Optional but powerful. |
| `user` | The human end-user | The question or request. This is what someone types into the chat box. |
| `assistant` | The model itself | The model's replies. When you carry history, you replay past answers here so the model sees its own previous turn. |

A fourth role, `tool`, shows up in Week 2 when we do function calling. Ignore it for now.

The examples below use `user` and `assistant` only. We add `system` in Lab 2 and see how big a difference it makes.


**Your first LLM call from Python**

In [8]:
response = ollama.chat(
    model="llama3.2:1b",
    messages=[{"role": "user", "content": "Hi, I am Serge."}],
)
response.model_dump()

{'model': 'llama3.2:1b',
 'created_at': '2026-08-13T16:35:14.451860699Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 767017081,
 'load_duration': 200768582,
 'prompt_eval_count': 31,
 'prompt_eval_duration': 49860000,
 'eval_count': 11,
 'eval_duration': 514870000,
 'message': {'role': 'assistant',
  'content': 'Hello Serge. How can I assist you today?',
  'thinking': None,
  'images': None,
  'tool_name': None,
  'tool_calls': None},
 'logprobs': None}

In [ ]:
print("First call:", response["message"]["content"])


### **Step 5 : a fresh new call. Ask what your name is.**

In [ ]:
response = ollama.chat(
    model="llama3.2:1b",
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("\nNew call (no history sent):", response["message"]["content"])
# Notice: the model has no idea who you are. Each call is independent.

### **Step 6 : carry the history yourself**

In [ ]:
messages = [
    {"role": "user",      "content": "Hi, I am Serge."},
    {"role": "assistant", "content": "Hello Serge! Nice to meet you."},
    {"role": "user",      "content": "What is my name?"},
]
response = ollama.chat(model="llama3.2:1b", messages=messages)
print("\nSame question with history:", response["message"]["content"])


Same question with history: I can't reveal your identity. Is there anything else I can help you with?


> **Some answers the model will generate**
* "*I don't have any information about a person named Serge ...*": The model was able to understand the "Serge" is the name, but it is not able to formulate the answer
* "*I can't reveal your identity. Is there anything else I can help you with?*": The model think we are trying to extract PII informations

> **Why this happens? Why is the model not able to answer with 'Serge'?**
>
> That can happen, especially with a small model like `llama3.2:1b`. LLMs are probabilistic:
> they sample the next word from a probability distribution, not from a deterministic rule.
> Two runs of the exact same call can produce different answers.
>
> If the model says "I do not know your name" or invents a different name, try again. Run the
> cell 3 or 4 times. Most of the time you will see "Serge" appear because the greeting is
> right there in the messages list. Sometimes the model will still miss it. That is the
> honest reality of small models. Bigger models (Claude, GPT) get this right almost every time.
>
> When we deploy Claude in Lab 5 you will see a big jump in reliability. Same code, better model.


**The importance of well-crafted prompts**

In [11]:
messages = [
    {"role": "user",      "content": "Hi, I am Serge."},
    {"role": "assistant", "content": "Hello Serge! Nice to meet you."},
    {"role": "user",      "content": "What is my last message?"},
]
response = ollama.chat(model="llama3.2:1b", messages=messages)
print("\nThe model is able to remember the last message:", response["message"]["content"])


The model is able to remember the last message: Your last message was "Hi, I am Serge".


In [12]:
messages = [
    {"role": "user",      "content": "Hi, I am Serge."},
    {"role": "assistant", "content": "Hello Serge! Nice to meet you."},
    {"role": "user",      "content": "From my last message, what is my name?"},
]
response = ollama.chat(model="llama3.2:1b", messages=messages)
print("\nNow the model is able to extract the name:", response["message"]["content"])


Now the model is able to extract the name: Your name is Serge. I remember now, we started a conversation with your greeting "Hi, I am Serge".


# Lab 2: System prompts (the model's standing orders)

**Objective.** See that the SAME user message produces VERY different answers depending on the system prompt. The system prompt is where the product lives.

**Prerequisites.** Lab 1 complete. Ollama running. `llama3.2:1b` pulled.

**What you will do.**

1. Send "Hi, I am Serge." with a POLITE system prompt.
2. Send the exact same user message with a PIRATE system prompt.
3. Compare the two outputs side by side.

**What you should see.**

- Polite output: "Welcome Serge! How can I help you today?"
- Pirate output: "Arr Serge! Welcome aboard, matey!"

**Why this matters.** Notion AI, Copy.ai, Cursor, and Jasper are each ONE chat LLM plus a tuned system prompt. The user only ever sees a chat box. 80% of "prompt engineering" is writing a good system prompt. This lab shows you why.

In [ ]:
import ollama

# The "system" message sets WHO the model is and HOW it behaves.
# The "user" message is just your question.
messages = [
    {"role": "system", "content": "You are a polite assistant. Greet the user by name and reply in two short sentences."},
    {"role": "user",   "content": "Hi, I am Serge."},
]
response = ollama.chat(model="llama3.2:1b", messages=messages)
print("Polite version:\n", r["message"]["content"])

# Same user message, different system prompt = different personality.
messages = [
    {"role": "system", "content": "You are a 17th-century pirate. Speak only in pirate slang. Mention the sea at least once."},
    {"role": "user",   "content": "Hi, I am Serge."},
]
response = ollama.chat(model="llama3.2:1b", messages=messages)
print("\nPirate version:\n", r["message"]["content"])
# System prompt = standing orders.  User message = the ask.

# Lab 3 : Temperature (the creativity dial)

**Objective.** Feel what the temperature dial does. Same prompt, three temperatures, three very different outputs.

**Prerequisites.** Lab 1 complete. Ollama running.

**What you will do.**

1. Send "Write a one-line tagline for a coffee shop called Serge's Brews." with `temperature=0.0`.
2. Send the same prompt with `temperature=0.7`.
3. Send the same prompt with `temperature=1.5`.
4. Run each cell twice. Notice which run repeats itself and which one changes.

**What you should see.**

- `temp=0.0`: same tagline every time (deterministic).
- `temp=0.7`: slightly different each run (balanced default).
- `temp=1.5`: wild, sometimes off-topic.

**Why this matters.** The three production presets you will use for the rest of your career:

- `temp=0.0` for code, math, classifiers, JSON extraction (reproducible).
- `temp=0.7` for chat apps and customer support (balanced default).
- `temp=1.5` for brainstorming only. Never production.

The other knobs you will meet in Week 2: `max_tokens`, `top_p`, `frequency_penalty`.

In [ ]:
import ollama

question = "Write a one-line tagline for a coffee shop called Serge's Brews."

for temp in [0.0, 0.7, 1.5]:
    response = ollama.chat(
        model="llama3.2:1b",
        messages=[{"role": "user", "content": question}],
        options={"temperature": temp},
    )
    print(f"\n--- temperature={temp} ---")
    print(response["message"]["content"])

# Lab 4 : Streaming (the ChatGPT effect)

**Objective.** Ask a big question two ways: without streaming, then with streaming. Watch the wall-clock time and the FELT time. Learn why streaming matters for your infrastructure.

**Prerequisites.** Lab 1 complete. Ollama running.

**What you will do.**

1. Ask "Explain RAG to a CFO in 5 paragraphs." with `stream=False`. Time it.
2. Ask the exact same question with `stream=True`. Print tokens as they arrive.
3. Compare the wall-clock time AND how it FELT to wait.

**What you should see.**

- Run 1: about 8 seconds of silence, then the whole answer dumps at once.
- Run 2: tokens type out word by word, ChatGPT-style.
- Total time is similar. The FELT time is night and day better.

**Why this matters (the DevOps angle).**

- Non-streaming for 30 seconds means your ALB or API Gateway idle timeout trips.
- Streaming keeps data flowing which means no timeouts.
- You will configure those timeouts in Week 7 (Docker) and Week 8 (deploy).

In [ ]:
import ollama
import time

question = "Write 5 short paragraphs explaining what RAG is to a CFO."

In [ ]:
print("Calling WITHOUT streaming...")
t0 = time.time()
response = ollama.chat(
    model="llama3.2:1b",
    messages=[{"role": "user", "content": question}],
)
print(f"Waited {time.time()-t0:.1f}s in silence. Then:\n")
print(response["message"]["content"])

In [ ]:
print("\n\nCalling WITH streaming...")
t0 = time.time()
for chunk in ollama.chat(
    model="llama3.2:1b",
    messages=[{"role": "user", "content": question}],
    stream=True,
):
    print(chunk["message"]["content"], end="", flush=True)
print(f"\n\nDone in {time.time()-t0:.1f}s. Same total time, very different feel.")